<a href="https://colab.research.google.com/github/AdityaaaTiwari/AdityaaaTiwari/blob/main/DataEngineer_Assesment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import numpy as np
import json
import hashlib
from datetime import datetime, timedelta

In [2]:
end_date=datetime.utcnow()

start_date=end_date-timedelta(days=30)

start=start_date.strftime('%Y-%m-%d')

end=end_date.strftime('%Y-%m-%d')

print(start_date)
print(end_date)

2026-04-27 17:37:44.279038
2026-05-27 17:37:44.279038


/tmp/ipykernel_5490/2199694847.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date=datetime.utcnow()


In [3]:
url=f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={start_date}&endtime={end_date}&minmagnitude=2.5&orderby=time"

print(url)

https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-04-27 17:37:44.279038&endtime=2026-05-27 17:37:44.279038&minmagnitude=2.5&orderby=time


In [4]:
response = requests.get(url)

raw_data = response.json()

In [5]:
print("Total Records:", len(raw_data["features"]))

Total Records: 1781


In [6]:
type(raw_data)

dict

In [7]:
with open("raw_earthquakes.json", "w") as file:
    json.dump(
        raw_data,
        file,
        indent=4)

print("raw json saved")

raw json saved


In [8]:
raw_hash = hashlib.md5(
    json.dumps(
        raw_data,
        sort_keys=True
        ).encode()
).hexdigest()

print(raw_hash)

00e45391dbd3bde9e459075eb77fb5c5


In [9]:
raw_data.keys()

dict_keys(['type', 'metadata', 'features', 'bbox'])

In [10]:
len(
raw_data['features']
)

1781

In [11]:
#inspect one event:
raw_data['features'][1]

{'type': 'Feature',
 'properties': {'mag': 6,
  'place': 'western Indian-Antarctic Ridge',
  'time': 1779894071567,
  'updated': 1779903104379,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/us7000snvt',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us7000snvt&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': 0,
  'alert': 'green',
  'status': 'reviewed',
  'tsunami': 0,
  'sig': 554,
  'net': 'us',
  'code': '7000snvt',
  'ids': ',us7000snvt,',
  'sources': ',us,',
  'types': ',losspager,moment-tensor,origin,phase-data,shakemap,',
  'nst': 46,
  'dmin': 9.39,
  'rms': 0.66,
  'gap': 68,
  'magType': 'mww',
  'type': 'earthquake',
  'title': 'M 6.0 - western Indian-Antarctic Ridge'},
 'geometry': {'type': 'Point', 'coordinates': [139.3037, -50.5213, 10]},
 'id': 'us7000snvt'}

Understand required columns

Create container

In [12]:
records=[]

Flatten nested JSON

In [13]:
# Coordinates extraction is mandatory:
for item in raw_data["features"]:

    properties = item["properties"]
    geometry = item["geometry"]

    coordinates = geometry["coordinates"]

    earthquake = {

        "event_id": item["id"],

        "magnitude": properties.get("mag"),

        "place": properties.get("place"),

        "event_time": properties.get("time"),

        "updated_time": properties.get("updated"),

        "tsunami": properties.get("tsunami"),

        "significance": properties.get("sig"),

        "alert": properties.get("alert"),

        "status": properties.get("status"),

        "event_type": properties.get("type"),

        "longitude": coordinates[0],

        "latitude": coordinates[1],

        "depth_km": coordinates[2]
    }

    records.append(earthquake)

Convert into DataFrame

In [14]:
df=pd.DataFrame(records)

In [15]:
df.head()

,event_id,magnitude,place,event_time,updated_time,tsunami,significance,alert,status,event_type,longitude,latitude,depth_km
0,ci41475480,2.57,"21 km N of Searles Valley, CA",1779900430740,1779903060304,0,102,None,reviewed,earthquake,-117.388167,35.954333,2.58
1,us7000snvt,6.00,western Indian-Antarctic Ridge,1779894071567,1779903104379,0,554,green,reviewed,earthquake,139.303700,-50.521300,10.00
2,us7000snvr,6.00,western Indian-Antarctic Ridge,1779893464419,1779901905940,0,554,green,reviewed,earthquake,139.367800,-50.517400,10.00
3,aka2026kkqqsb,2.50,"44 km WNW of Anchor Point, Alaska",1779885791554,1779892503040,0,96,None,automatic,earthquake,-152.570000,59.924000,87.80
4,us7000snvf,2.50,"10 km NNE of Damar, Kansas",1779885787860,1779900052317,0,96,None,reviewed,earthquake,-99.532500,39.405800,9.90


Inspect DataFrame shape

In [16]:
#Before cleaning, Data Engineers inspect data size
print("Rows and Columns:")

df.shape

Rows and Columns:


(1781, 13)

In [17]:
#Check column names
df.columns

Index(['event_id', 'magnitude', 'place', 'event_time', 'updated_time',
       'tsunami', 'significance', 'alert', 'status', 'event_type', 'longitude',
       'latitude', 'depth_km'],
      dtype='object')

Check missing values BEFORE cleaning

In [18]:
missing_before=df.isnull().sum()

print(missing_before)


event_id           0
magnitude          0
place              0
event_time         0
updated_time       0
tsunami            0
significance       0
alert           1715
status             0
event_type         0
longitude          0
latitude           0
depth_km           0
dtype: int64


missing value rules![Screenshot 2026-05-27 142623.png](https://raw.githubusercontent.com/jupyter-resources/jupyter-resources/main/resources/Screenshot%202026-05-27%20142623.png)

In [19]:
df['place']=df['place'].fillna(
    "Unknown"
)

df['alert']=df['alert'].fillna(
    "none"
)

df.dropna(
subset=[

'event_id',

'magnitude',

'latitude',

'longitude',

'depth_km'

],
 inplace=True
)

Verify cleaning worked

In [51]:
missing_after = df.isnull().sum()

print(missing_after)

event_id          0
magnitude         0
place             0
event_time        0
updated_time      0
tsunami           0
significance      0
alert             0
status            0
event_type        0
longitude         0
latitude          0
depth_km          0
place_clean       0
event_year        0
event_month       0
event_day         0
event_hour        0
risk_level        0
depth_category    0
region            0
dtype: int64


Remove duplicates

In [54]:
before = df.shape[0]

df.drop_duplicates(subset=["event_id"], inplace=True)

after = df.shape[0]

duplicates_removed = before - after

print(duplicates_removed)

0


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1781 entries, 0 to 1780
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   event_id        1781 non-null   object        
 1   magnitude       1781 non-null   float64       
 2   place           1781 non-null   object        
 3   event_time      1781 non-null   datetime64[ns]
 4   updated_time    1781 non-null   datetime64[ns]
 5   tsunami         1781 non-null   int64         
 6   significance    1781 non-null   int64         
 7   alert           1781 non-null   object        
 8   status          1781 non-null   object        
 9   event_type      1781 non-null   object        
 10  longitude       1781 non-null   float64       
 11  latitude        1781 non-null   float64       
 12  depth_km        1781 non-null   float64       
 13  place_clean     1781 non-null   object        
 14  event_year      1781 non-null   int32         
 15  even

In [56]:
df["event_time"] = pd.to_datetime(
    df["event_time"],
    unit="ms",
    errors="coerce"
)

df["updated_time"] = pd.to_datetime(
    df["updated_time"],
    unit="ms",
    errors="coerce"
)

df["magnitude"] = df["magnitude"].astype(float)

df["longitude"] = df["longitude"].astype(float)

df["latitude"] = df["latitude"].astype(float)

df["depth_km"] = df["depth_km"].astype(float)

df["tsunami"] = df["tsunami"].astype(int)

df["significance"] = df["significance"].astype(int)

In [59]:
df.dtypes

,0
event_id,object
magnitude,float64
place,object
event_time,datetime64[ns]
updated_time,datetime64[ns]
tsunami,int64
significance,int64
alert,object
status,object
event_type,object


In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1781 entries, 0 to 1780
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   event_id        1781 non-null   object        
 1   magnitude       1781 non-null   float64       
 2   place           1781 non-null   object        
 3   event_time      1781 non-null   datetime64[ns]
 4   updated_time    1781 non-null   datetime64[ns]
 5   tsunami         1781 non-null   int64         
 6   significance    1781 non-null   int64         
 7   alert           1781 non-null   object        
 8   status          1781 non-null   object        
 9   event_type      1781 non-null   object        
 10  longitude       1781 non-null   float64       
 11  latitude        1781 non-null   float64       
 12  depth_km        1781 non-null   float64       
 13  place_clean     1781 non-null   object        
 14  event_year      1781 non-null   int32         
 15  even

In [61]:
#Datetime conversion
df['event_time']=pd.to_datetime(

df['event_time'],

unit='ms',

errors='coerce'
)


df['updated_time']=pd.to_datetime(

df['updated_time'],

unit='ms',

errors='coerce'
)

verify data types

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1781 entries, 0 to 1780
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   event_id      1781 non-null   object        
 1   magnitude     1781 non-null   float64       
 2   place         1781 non-null   object        
 3   event_time    1781 non-null   datetime64[ns]
 4   updated_time  1781 non-null   datetime64[ns]
 5   tsunami       1781 non-null   int64         
 6   significance  1781 non-null   int64         
 7   alert         1781 non-null   object        
 8   status        1781 non-null   object        
 9   event_type    1781 non-null   object        
 10  longitude     1781 non-null   float64       
 11  latitude      1781 non-null   float64       
 12  depth_km      1781 non-null   float64       
dtypes: datetime64[ns](2), float64(4), int64(2), object(5)
memory usage: 181.0+ KB


Text Cleaning
Text standardization

In [62]:
text_columns = [
    "place",
    "alert",
    "status",
    "event_type"
]

for col in text_columns:

    df[col] = (
        df[col]

        .astype(str)

        .str.strip()

        .str.lower()
    )

In [64]:
df["place_clean"] = df["place"].str.title()

In [65]:
df[[
    "place",
    "place_clean"
    ]].head()

,place,place_clean
0,"21 km n of searles valley, ca","21 Km N Of Searles Valley, Ca"
1,western indian-antarctic ridge,Western Indian-Antarctic Ridge
2,western indian-antarctic ridge,Western Indian-Antarctic Ridge
3,"44 km wnw of anchor point, alaska","44 Km Wnw Of Anchor Point, Alaska"
4,"10 km nne of damar, kansas","10 Km Nne Of Damar, Kansas"


Create Date Features

In [66]:
df["event_year"] = df["event_time"].dt.year

df["event_month"] = df["event_time"].dt.month

df["event_day"] = df["event_time"].dt.day

df["event_hour"] = df["event_time"].dt.hour

In [67]:
df[
    [
        "event_time",
        "event_year",
        "event_month",
        "event_day",
        "event_hour"
    ]
].head()

,event_time,event_year,event_month,event_day,event_hour
0,2026-05-27 16:47:10.740,2026,5,27,16
1,2026-05-27 15:01:11.567,2026,5,27,15
2,2026-05-27 14:51:04.419,2026,5,27,14
3,2026-05-27 12:43:11.554,2026,5,27,12
4,2026-05-27 12:43:07.860,2026,5,27,12


**Feature** **Engineering**

Create Risk Level
HIGH:

magnitude ≥6
OR tsunami=1
OR alert = yellow/orange/red

MEDIUM:

magnitude between 4.5–6
OR significance≥600

Else:

LOW

In [69]:
def risk_level(row):


    if (
        row["magnitude"] >= 6.0
        or row["tsunami"] == 1
        or row["alert"] in ["yellow", "orange", "red"]
    ):

        return "high"

    elif (
        4.5 <= row["magnitude"] < 6.0
        or row["significance"] >= 600
    ):

        return "medium"

    else:

        return "low"

In [70]:
df["risk_level"] = df.apply(risk_level, axis=1)

df[
    [
        "magnitude",
        "alert",
        "tsunami",
        "risk_level"
    ]
].head()

,magnitude,alert,tsunami,risk_level
0,2.57,none,0,low
1,6.00,green,0,high
2,6.00,green,0,high
3,2.50,none,0,low
4,2.50,none,0,low


In [71]:
def depth_category(depth):

    if depth < 70:

        return "Shallow"

    elif depth <= 300:

        return "Intermediate"

    else:

        return "Deep"

In [72]:
df["depth_category"] = df["depth_km"].apply(depth_category)

df[
    [
        "depth_km",
        "depth_category"
    ]
].head()

,depth_km,depth_category
0,2.58,Shallow
1,10.00,Shallow
2,10.00,Shallow
3,87.80,Intermediate
4,9.90,Shallow


In [73]:
def extract_region(place):

    if "," in place:

        return place.split(",")[-1].strip()

    else:

        return "Unknown"

In [74]:
df["region"] = df["place"].apply(extract_region)

df[
    [
        "place",
        "region"
    ]
].head()

,place,region
0,"21 km n of searles valley, ca",ca
1,western indian-antarctic ridge,Unknown
2,western indian-antarctic ridge,Unknown
3,"44 km wnw of anchor point, alaska",alaska
4,"10 km nne of damar, kansas",kansas


Quick verification


In [75]:
df.columns

Index(['event_id', 'magnitude', 'place', 'event_time', 'updated_time',
       'tsunami', 'significance', 'alert', 'status', 'event_type', 'longitude',
       'latitude', 'depth_km', 'place_clean', 'event_year', 'event_month',
       'event_day', 'event_hour', 'risk_level', 'depth_category', 'region'],
      dtype='object')

Output Files

In [77]:
df.to_csv(
    "earthquakes_clean.csv",
    index=False
)

print("CSV file saved")

CSV file saved


In [79]:
df.to_parquet(

'earthquakes_clean.parquet',

index=False

)

In [80]:
with open(
    "raw_earthquakes.json",
    "w"
) as file:

    json.dump(
        raw_data,
        file,
        indent=4
    )

Create risk counts

In [45]:
high_risk_count=(

df['risk_level']=='high'

).sum()


medium_risk_count=(

df['risk_level']=='medium'

).sum()


low_risk_count=(

df['risk_level']=='low'

).sum()

In [82]:
# cheking
print(high_risk_count)

print(medium_risk_count)

print(low_risk_count)

11
404
1366


Top 5 regions


In [47]:
top_regions=(

df['region']

.value_counts()

.head(5)

.to_dict()

)

In [48]:
top_regions

{'Alaska': 556, 'Unknown': 134, 'Ca': 129, 'Puerto Rico': 79, 'Japan': 66}

Data Quality **Report** **bold text**
![image.png](https://raw.githubusercontent.com/jupyter-resources/jupyter-resources/main/resources/image.png)

In [83]:
quality_report={

"data_fetched_at_utc":

str(datetime.utcnow()),


"raw_record_count":

len(
raw_data[
'features'
]
),


"clean_record_count":

len(df),


"duplicates_removed":

duplicates_removed,


"missing_values_before":

missing_before.to_dict(),


"missing_values_after":

missing_after.to_dict(),


"high_risk_events":

high_risk_count,


"medium_risk_events":

medium_risk_count,


"low_risk_events":

low_risk_count,


"max_magnitude":

df['magnitude'].max(),


"deepest_earthquake_km":

df['depth_km'].max(),


"top_5_regions_by_event_count":

top_regions

}

/tmp/ipykernel_5490/2415161062.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  str(datetime.utcnow()),


In [86]:
with open("data_quality_report.json", "w") as file:

    json.dump(
        quality_report,
        file,
        indent=4
    )

print("Quality report saved")

Quality report saved


In [87]:
quality_report

{'data_fetched_at_utc': '2026-05-27 17:52:28.029398',
 'raw_record_count': 1781,
 'clean_record_count': 1781,
 'duplicates_removed': 0,
 'missing_values_before': {'event_id': 0,
  'magnitude': 0,
  'place': 0,
  'event_time': 0,
  'updated_time': 0,
  'tsunami': 0,
  'significance': 0,
  'alert': 1715,
  'status': 0,
  'event_type': 0,
  'longitude': 0,
  'latitude': 0,
  'depth_km': 0},
 'missing_values_after': {'event_id': 0,
  'magnitude': 0,
  'place': 0,
  'event_time': 0,
  'updated_time': 0,
  'tsunami': 0,
  'significance': 0,
  'alert': 0,
  'status': 0,
  'event_type': 0,
  'longitude': 0,
  'latitude': 0,
  'depth_km': 0,
  'place_clean': 0,
  'event_year': 0,
  'event_month': 0,
  'event_day': 0,
  'event_hour': 0,
  'risk_level': 0,
  'depth_category': 0,
  'region': 0},
 'high_risk_events': 11,
 'medium_risk_events': 404,
 'low_risk_events': 1366,
 'max_magnitude': 6.9,
 'deepest_earthquake_km': 639.858,
 'top_5_regions_by_event_count': {'Alaska': 556,
  'Unknown': 134,
 

Save report as JSON

In [88]:
import numpy as np

for key, value in quality_report.items():
    if isinstance(value, np.int64):
        quality_report[key] = int(value)

with open(
'quality_report.json',
'w'
) as file:
    json.dump(
    quality_report,
    file,
    indent=4
)

In [89]:
#Which region had most earthquakes?
most_region=(

df['region']

.value_counts()

.idxmax()

)

print(

"Region with most earthquakes:",

most_region

)

Region with most earthquakes: alaska


In [90]:
#Strongest earthquake:
strongest=df.loc[

df['magnitude']

.idxmax()

]

print(

strongest[
'magnitude'
]

)

print(

strongest[
'place_clean'
]

)

6.9
29 Km Ene Of Calama, Chile


In [91]:
#High-risk events:
print(

"High Risk Events:",

high_risk_count

)

High Risk Events: 11


In [92]:
#Depth categories:
df[
'depth_category'
].value_counts()

,count
depth_category,
Shallow,1416
Intermediate,295
Deep,70


In [93]:
#Most active earthquake day:
busy_day=(

df['event_day']

.value_counts()

.idxmax()

)

print(

"Most active day:",

busy_day

)

Most active day: 10


Final file check

In [94]:
!ls

data_quality_report.json  earthquakes_clean.parquet  raw_earthquakes.json
earthquakes_clean.csv	  quality_report.json	     sample_data


Save Final Data Quality Report

In [96]:
with open(
    "final_data_quality_report.json",
    "w"
) as file:

    json.dump(
        quality_report,
        file,
        indent=4
    )

In [97]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
